# EDA Framework

For future datasets, remember this sequence:

**Understand → Validate → Clean → Describe → Visualize → Compare → Investigate → Explain → Recommend**

A data analyst should not stop at *"sales decreased."* Continue asking:

**Where? → When? → Which segment/product/customer? → What variables are associated with it? → Is it an anomaly or recurring pattern? → What should the business investigate or do next?**



## Step 1 — Import libraries

In [60]:
import pandas as pd
import plotly.express as px
from IPython.display import display

## Step 2 — Load the dataset


In [61]:
df = pd.read_csv("listings.csv")


**Ask:** What does one row represent? How many records and columns do we have?

In [62]:
df.shape

(22708, 90)

In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            22708 non-null  int64  
 1   listing_url                                   22708 non-null  str    
 2   scrape_id                                     22708 non-null  int64  
 3   last_scraped                                  22708 non-null  str    
 4   source                                        22708 non-null  str    
 5   name                                          22708 non-null  str    
 6   description                                   21823 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   22707 non-null  str    
 9   host_id                                       22708 non-null  int64  
 1

## Step 3 — Understand structure and data types

**Ask:** Are numeric fields numeric? Is the date actually stored as a date?


In [64]:
# Keep your original selection, but use names so column order cannot change it.
cols_to_drop = [
    'listing_url', 'neighborhood_overview', 'picture_url', 'host_profile_url',
    'host_since', 'host_about', 'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_thumbnail_url', 'host_picture_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'neighbourhood', 'price_quote_price_per_night', 'price_quote_raw',
    'calendar_updated', 'instant_bookable'
]
df = df.drop(columns=cols_to_drop)
# Dropping a column is a scope choice; it does not mean that column is invalid.

**Convert** data to proper type

In [65]:
# Remove currency formatting; keep missing/unreadable prices as missing.
df['price'] = pd.to_numeric(
    df['price'].replace(r'[\$,]', '', regex=True), errors='coerce'
)

In [66]:
df['price'].dtypes

dtype('float64')

In [67]:
display(df.sample(5, random_state=42))
df.info()

,id,scrape_id,last_scraped,source,name,description,host_id,host_url,host_profile_id,host_name,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
20867,1241902600240413011,20260620164952,2026-06-21,city scrape,Wanderlust Gran Vía Apartments,Enjoy a luxury experience in this central acco...,265228558,https://www.airbnb.com/users/show/265228558,1.463338e+18,Hernando,...,4.17,4.50,4.92,4.17,NaN,2,2,0,0,0.67
22212,1187669688678088715,20260620164952,2026-06-22,city scrape,CituSpace Las Tablas-1 Bed with sofa bed & Par...,NaN,31772865,https://www.airbnb.com/users/show/31772865,1.465467e+18,Cituspace,...,4.40,4.40,4.60,4.60,Spain - National registration number<br />ESFC...,74,74,0,0,0.64
3108,1293407313235376262,20260620164952,2026-06-21,city scrape,Méndez Álvaro 1 Dormitorio,Discover a new way to experience Madrid at the...,662112418,https://www.airbnb.com/users/show/662112418,1.470662e+18,Plenora,...,4.91,4.77,4.67,4.54,Spain - National registration number<br />ESFC...,2,2,0,0,8.63
15447,891999910962966184,20260620164952,2026-07-02,previous scrape,Cómoda Habitación Madrid,Ideal room for 1 person.<br />Well lit with a ...,131659268,https://www.airbnb.com/users/show/131659268,1.468318e+18,Jorge,...,5.00,5.00,5.00,4.93,NaN,1,0,1,0,0.37
12049,45320031,20260620164952,2026-07-02,previous scrape,Habitacion para noches O dias.,NaN,365289537,https://www.airbnb.com/users/show/365289537,1.469735e+18,Fulmate,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN


<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 71 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            22708 non-null  int64  
 1   scrape_id                                     22708 non-null  int64  
 2   last_scraped                                  22708 non-null  str    
 3   source                                        22708 non-null  str    
 4   name                                          22708 non-null  str    
 5   description                                   21823 non-null  str    
 6   host_id                                       22708 non-null  int64  
 7   host_url                                      22708 non-null  str    
 8   host_profile_id                               22556 non-null  float64
 9   host_name                                     22556 non-null  str    
 1

In [68]:
# convert them to datetime format
# [last_scraped, price_quote_checkin_date, price_quote_checkout_date, calendar_last_scraped, first_review, last_review]
date_columns = [
    'last_scraped',
    'price_quote_checkin_date',
    'price_quote_checkout_date',
    'calendar_last_scraped',
    'first_review',
    'last_review'
]

df[date_columns] = df[date_columns].apply(pd.to_datetime, errors='coerce')

# ".apply()" is doing the function for each column consequently, working as for loop

In [69]:
df[date_columns].dtypes

last_scraped                 datetime64[us]
price_quote_checkin_date     datetime64[us]
price_quote_checkout_date    datetime64[us]
calendar_last_scraped        datetime64[us]
first_review                 datetime64[us]
last_review                  datetime64[us]
dtype: object

## Step 4 — Check missing values

**Ask:** Which columns have missing values, and what percentage is missing?


In [70]:
location_null_listings = df['host_location'].isnull().index

#df.loc('columns'= [])

In [71]:
# Multiply by 100 to show percentages rather than proportions.
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
})
missing[missing['missing_count'] > 0].sort_values(
    'missing_percent', ascending=False
).round(2)

,missing_count,missing_percent
license,8056,35.48
host_location,7485,32.96
bedrooms,5719,25.18
bathrooms,5405,23.80
beds,4544,20.01
review_scores_value,3718,16.37
review_scores_location,3717,16.37
review_scores_cleanliness,3716,16.36
review_scores_communication,3716,16.36
review_scores_checkin,3716,16.36


## Step 5 — Check duplicates and uniqueness

**Ask:** Are there duplicated rows? 


In [72]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate listing IDs:', df['id'].duplicated().sum())

Duplicate rows: 0
Duplicate listing IDs: 0


## Step 6 — Clean obvious data-quality issues

For practice, we will:
- remove exact duplicate rows;
- fill missing categorical values with `"Unknown"`;

In [73]:
df = df.drop_duplicates()
category_columns = ['room_type', 'property_type', 'neighbourhood_cleansed']
# Assign back to df: inplace=True on df[[...]] does not update df reliably.
df[category_columns] = df[category_columns].fillna('Unknown')
# Leave missing numerical values as missing; zero would change their meaning.

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22708 entries, 0 to 22707
Data columns (total 71 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   id                                            22708 non-null  int64         
 1   scrape_id                                     22708 non-null  int64         
 2   last_scraped                                  22708 non-null  datetime64[us]
 3   source                                        22708 non-null  str           
 4   name                                          22708 non-null  str           
 5   description                                   21823 non-null  str           
 6   host_id                                       22708 non-null  int64         
 7   host_url                                      22708 non-null  str           
 8   host_profile_id                               22556 non-null  float64       


## Step 7 — Descriptive statistics

**Ask:** What are the typical values, ranges, and variability?


In [ ]:
# IDs are numbers, but their averages and correlations are not meaningful.
numeric_cols = [
    'price', 'accommodates', 'bedrooms', 'bathrooms', 'beds',
    'review_scores_rating', 'number_of_reviews', 'availability_365',
    'minimum_nights', 'estimated_revenue_l365d', 'estimated_occupancy_l365d', 'reviews_per_month'
]
df[numeric_cols].describe().round(2)
df.select_dtypes(include= 'number')

,id,scrape_id,host_id,host_profile_id,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_listings_count,latitude,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1008521039996399925,20260620164952,509530871,1.470291e+18,3.0,2.0,2.0,7.0,1.0,40.387990,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
1,887679074200493161,20260620164952,425181129,1.469928e+18,4.0,8.0,3.0,1.0,1.0,40.385482,...,5.00,5.00,5.00,5.00,5.00,1,1,0,0,0.03
2,21376743,20260620164952,43556711,1.465647e+18,10.0,9.0,8.0,8.0,2.0,40.422940,...,4.25,4.75,4.75,5.00,4.75,1,1,0,0,0.04
3,1546256895864909953,20260620164952,726634087,NaN,NaN,NaN,NaN,NaN,NaN,40.422150,...,5.00,5.00,4.00,5.00,5.00,7,0,7,0,0.24
4,1542083092059647802,20260620164952,726323161,NaN,NaN,NaN,NaN,NaN,NaN,40.424880,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22703,1594902380071464129,20260620164952,739050243,NaN,NaN,NaN,NaN,NaN,NaN,40.481870,...,4.92,4.84,4.76,4.78,4.74,9,0,9,0,11.28
22704,1597629271761937213,20260620164952,739050243,NaN,NaN,NaN,NaN,NaN,NaN,40.481870,...,4.95,4.86,4.71,4.86,4.67,9,0,9,0,4.63
22705,1594902380014662811,20260620164952,739050243,NaN,NaN,NaN,NaN,NaN,NaN,40.481866,...,4.93,4.87,4.85,4.57,4.65,9,0,9,0,13.53
22706,1594902379987415877,20260620164952,739050243,NaN,NaN,NaN,NaN,NaN,NaN,40.481870,...,4.82,4.82,4.64,4.64,4.55,9,0,9,0,2.39


In [76]:
df['source'].unique()

<StringArray>
['city scrape', 'previous scrape']
Length: 2, dtype: str

In [77]:
df[category_columns].describe()

,room_type,property_type,neighbourhood_cleansed
count,22708,22708,22708
unique,4,59,128
top,Entire home/apt,Entire rental unit,Embajadores
freq,15899,14291,2453


## Step 8 — Explore categorical variables

**Ask:** Which room types, property types, and neighbourhoods have the most listings?
The charts show up to 12 categories by listing count, not by price.

In [78]:
categorical_cols = category_columns

In [79]:
df['property_type'].value_counts()

property_type
Entire rental unit                    14291
Private room in rental unit            4535
Private room in home                    565
Entire condo                            433
Room in hotel                           427
Entire loft                             384
Private room in hostel                  327
Entire serviced apartment               309
Entire home                             291
Private room in condo                   235
Private room in bed and breakfast       118
Room in aparthotel                       78
Private room in guesthouse               72
Private room in casa particular          67
Shared room in bed and breakfast         63
Private room in guest suite              49
Room in boutique hotel                   37
Private room                             35
Shared room in hotel                     31
Room in hostel                           30
Shared room in hostel                    27
Private room in serviced apartment       27
Entire guesthouse 

**Visualize** the categorical columns

In [80]:
for col in categorical_cols:
    value_counts = df[col].value_counts().head(12).reset_index()
    value_counts.columns = [col, 'Count']
    fig = px.bar(
        value_counts, x=col, y='Count', text_auto=True,
        title=f'{col} — up to 12 most common categories'
    )
    fig.show()

### Review of your work through step 8

**What works well:** You load and inspect the dataset, convert price and dates,
check missing values and duplicates, describe the data, and count categories.
This is a good EDA sequence. One row represents a listing; listing IDs are unique
in this file.

**Small fixes made:**
- Step 3 now drops columns by name and explicitly displays the sample. Your date
  conversion with `.apply(pd.to_datetime, errors='coerce')` is appropriate.
- Step 4 reports missing percentages: `.mean()` alone gives a fraction.
- Step 5 checks repeated listing IDs as well as repeated rows.
- Step 6 assigns `fillna()` back to the dataframe, fixing the saved chained-assignment
  warning. Missing numerical values are left missing.
- Step 7 focuses on useful measurements instead of IDs and summarizes categories.
- Step 8 keeps your frequency charts, with titles explaining the top-12 limit.

**Next:** Compare numerical variables, then prices across categories. A numeric ID
is not a measurement. Missing prices are not free stays, and missing ratings are
not zero-star ratings. The dropped columns remain in the original CSV if needed.

## Step 9 — Explore numerical distributions

**Ask:** What is a typical price? Are there unusually large values?

Use `df` for the full dataset and `priced` for listings with a recorded positive
price. Prices stay in the dataset's recorded units; removing a currency symbol
does not convert currencies or establish the quote basis.

In [81]:
print('All listings:', len(df))
print('Missing prices:', df['price'].isna().sum())
print('Zero or negative prices:', df['price'].le(0).sum())
priced = df[df['price'] > 0].copy()
print('Listings available for price analysis:', len(priced))
display(priced['price'].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

All listings: 22708
Missing prices: 3536
Zero or negative prices: 0
Listings available for price analysis: 19172


count    19172.00
mean       180.06
std        555.62
min          3.36
25%         83.74
50%        125.00
75%        184.00
95%        399.21
99%        919.99
max      36883.00
Name: price, dtype: float64

In [82]:
# Limit ONLY this histogram to the 99th percentile so typical prices are visible.
price_limit = priced['price'].quantile(0.99)
price_view = priced[priced['price'] <= price_limit]
print('Histogram upper limit:', round(price_limit, 2))
print('Listings above this limit (kept in all summaries):', len(priced) - len(price_view))
px.histogram(
    price_view, x='price', nbins=50, text_auto=True,
    title='Price distribution — up to the 99th percentile (chart only)'
).show()

for col in ['accommodates', 'bedrooms', 'review_scores_rating', 'number_of_reviews']:
    px.histogram(df, x=col, nbins=30, text_auto=True, title=f'Distribution of {col}').show()

Histogram upper limit: 919.99
Listings above this limit (kept in all summaries): 192


**Read the output:** A mean much higher than the median suggests a long right tail.
Check each column's `count`: pandas omits missing values from these summaries.

## Step 10 — Flag possible price outliers using IQR

IQR is the distance from the 25th to the 75th percentile. Values outside
`Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR` are worth inspecting, not automatically deleting.
Large properties or different room types may naturally have different prices.

In [83]:
df['price'][df['price'] <=0]

Series([], Name: price, dtype: float64)

In [84]:
q1 = priced['price'].quantile(0.25)
q3 = priced['price'].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
price_outliers = priced[
    (priced['price'] < lower_limit) | (priced['price'] > upper_limit)
]
print('IQR limits:', round(lower_limit, 2), 'to', round(upper_limit, 2))
print('Flagged listings:', len(price_outliers))
print('Percent of priced listings:', round(len(price_outliers) / len(priced) * 100, 2))
px.box(
    priced, y='price', points=False, log_y=True,
    title='All positive prices — logarithmic axis, individual points hidden'
).show()

IQR limits: -66.64 to 334.38
Flagged listings: 1369
Percent of priced listings: 7.14


## Step 11 — Numerical vs numerical relationships

**Ask:** Does price tend to increase with capacity, bedrooms, or bathrooms?
Is it associated with ratings, review counts, or available days?

Spearman correlation compares ranks and is useful for these skewed measurements.
Values near +1 mean the variables tend to rise together; values near -1 mean
they move in opposite directions. A value near 0 means a weak rank association.
Correlation does not show cause and effect. Missing values are excluded separately
for each pair, so check the pair counts too.

In [85]:
correlation = priced[numeric_cols].corr(method='spearman')
display(correlation['price'].drop('price').sort_values(ascending=False).round(3))

pair_counts = priced[numeric_cols].notna().astype(int)
pair_counts = pair_counts.T.dot(pair_counts)
display(pair_counts['price'].rename('listings_with_both_values').to_frame())

px.imshow(
    correlation, text_auto='.2f', color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, title='Spearman correlation — positive-price listings'
).show()

accommodates                 0.642
beds                         0.549
bedrooms                     0.462
estimated_revenue_l365d      0.355
bathrooms                    0.297
number_of_reviews            0.121
review_scores_rating         0.082
estimated_occupancy_l365d    0.043
availability_365             0.028
minimum_nights              -0.202
Name: price, dtype: float64

,listings_with_both_values
price,19172
accommodates,19172
bedrooms,14840
bathrooms,17029
beds,17888
review_scores_rating,16464
number_of_reviews,19172
availability_365,19172
minimum_nights,19171
estimated_revenue_l365d,19172


In [86]:
# Sample only the scatter plots to keep the notebook responsive.
# All calculations above and below use the full eligible data.
scatter_data = priced.sample(n=min(2000, len(priced)), random_state=42)
print('Scatter plot sample:', len(scatter_data), 'listings')
for col in ['accommodates', 'bedrooms', 'review_scores_rating']:
    px.scatter(
        scatter_data, x=col, y='price', color='room_type', opacity=0.4,
        log_y=True, hover_data=['neighbourhood_cleansed'],
        title=f'Price vs {col} — sample, logarithmic price axis'
    ).show()

Scatter plot sample: 2000 listings


**Read the charts:** The logarithmic price axis includes large prices while making
typical listings easier to see. Equal vertical distances represent equal price
ratios, not equal price differences. Points with missing x-values are not drawn.
Available days are not the same as occupancy: unavailable dates may be booked or blocked.

## Step 12 — Numerical vs categorical: prices across groups

**Ask:** How do prices differ by room type, neighbourhood, and property type?

Use `groupby()` to calculate median and mean prices, and show how many prices are
available. Median is less affected by very expensive listings. The complete
tables include every category; the neighbourhood/property charts show the 12
categories with the most listings, ordered by median price.

In [87]:
# Include unpriced listings in group counts, but exclude invalid prices from summaries.
group_data = df.copy()
group_data['price'] = group_data['price'].where(group_data['price'] > 0)
price_summaries = {}

for col in category_columns:
    summary = group_data.groupby(col).agg(
        listings=('id', 'size'),
        priced_listings=('price', 'count'),
        median_price=('price', 'median'),
        mean_price=('price', 'mean')
    )
    summary['price_coverage_percent'] = summary['priced_listings'] / summary['listings'] * 100
    summary = summary.sort_values('median_price', ascending=False)
    price_summaries[col] = summary
    print(col)
    display(summary.round(2))

    common_groups = df[col].value_counts().head(12).index
    chart_summary = summary.loc[common_groups].sort_values('median_price').reset_index()
    px.bar(
        chart_summary, x='median_price', y=col, orientation='h', text_auto='.2f',
        hover_data=['listings', 'priced_listings', 'mean_price', 'price_coverage_percent'],
        title=f'Median price by {col} — up to 12 most common groups', height=500
    ).show()

room_type


,listings,priced_listings,median_price,mean_price,price_coverage_percent
room_type,,,,,
Entire home/apt,15899,13936,144.0,202.80,87.65
Hotel room,39,29,104.0,107.70,74.36
Private room,6611,5065,64.0,121.21,76.61
Shared room,159,142,25.0,62.00,89.31


property_type


,listings,priced_listings,median_price,mean_price,price_coverage_percent
property_type,,,,,
Entire villa,8,4,867.34,923.38,50.00
Private room in floor,4,2,700.40,700.40,50.00
Private room in cottage,1,1,359.50,359.50,100.00
Entire chalet,9,8,319.28,531.41,88.89
Religious building,2,2,264.00,264.00,100.00
Shared room in condo,2,2,260.15,260.15,100.00
Casa particular,6,4,207.74,378.12,66.67
Entire serviced apartment,309,269,173.00,360.75,87.06
Entire home,291,223,168.00,323.47,76.63


neighbourhood_cleansed


,listings,priced_listings,median_price,mean_price,price_coverage_percent
neighbourhood_cleansed,,,,,
Recoletos,307,264,193.68,257.48,85.99
Jerónimos,89,81,170.04,227.39,91.01
Goya,367,294,168.67,209.24,80.11
Castellana,179,147,164.63,223.16,82.12
Sol,1240,1125,163.88,237.01,90.73
...,...,...,...,...,...
Aguilas,49,40,52.85,80.91,81.63
Campamento,25,20,50.50,86.40,80.00
Apostol Santiago,34,30,48.50,82.48,88.24


In [88]:
# Box plots show spread and overlap, which median bars cannot show.
for col in category_columns:
    common_groups = df[col].value_counts().head(12).index
    box_data = priced[priced[col].isin(common_groups)]
    px.box(
        box_data, x=col, y='price', points=False, log_y=True,
        title=f'Price spread by {col} — common groups, logarithmic axis', height=500
    ).show()

**Interpret carefully:** A category with few recorded prices has a less stable median.
Different price coverage may also make comparisons unrepresentative. Property type
often already includes room type in its label, so these are related categories.

## Step 13 — Compare neighbourhoods within the same room type

**Ask:** Do neighbourhood differences remain when we compare similar room types?

The original template suggested a time series. This file is a listing snapshot
collected on a few dates, not repeated observations of each listing's price.
`last_review` is a latest-review date, not a booking or revenue history. A price
trend or seasonal sales conclusion would therefore be misleading here.

Instead, use a simple pivot table. Show the 12 most common neighbourhoods and
hide cells with fewer than 30 recorded prices. This is a practical display rule,
not a guarantee of statistical reliability.

In [89]:
common_neighbourhoods = df['neighbourhood_cleansed'].value_counts().head(12).index
comparison = priced[priced['neighbourhood_cleansed'].isin(common_neighbourhoods)]
median_by_room = comparison.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values='price', aggfunc='median'
)
count_by_room = comparison.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values='price',
    aggfunc='count', fill_value=0
)
display(count_by_room)
median_by_room = median_by_room.where(count_by_room >= 30)
display(median_by_room.round(2))
px.imshow(
    median_by_room, text_auto='.0f', color_continuous_scale='Blues',
    title='Median price by neighbourhood and room type — at least 30 prices per cell',
    aspect='auto', height=550
).show()

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,144,0,103,3
Cortes,718,0,186,0
Cuatro Caminos,257,0,85,54
Embajadores,1630,8,387,23
Goya,257,0,37,0
Guindalera,289,0,43,12
Justicia,742,0,200,1
Palacio,1197,1,260,11
Palos de Moguer,227,0,75,3


room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,160.33,NaN,43.16,NaN
Cortes,169.00,NaN,109.54,NaN
Cuatro Caminos,141.24,NaN,70.67,25.0
Embajadores,138.76,NaN,72.00,NaN
Goya,181.00,NaN,55.00,NaN
Guindalera,140.00,NaN,64.00,NaN
Justicia,169.50,NaN,84.42,NaN
Palacio,157.60,NaN,93.00,NaN
Palos de Moguer,139.00,NaN,69.00,NaN


Blank cells mean insufficient observations, not zero price. Comparing within room
type improves context, but property size, amenities, quote dates, and other
differences can still explain price gaps.

In [90]:
df['availability_365'].sort_values(ascending=True)
df['availability_365'][df['availability_365'] == 0].sort_values(ascending=True)

2        0
3        0
5        0
6        0
7        0
        ..
22698    0
22699    0
22700    0
22701    0
22702    0
Name: availability_365, Length: 2817, dtype: int64

## Step 14 — Inspect extremes and missing-price patterns

**Ask:** Which listings need a closer look? Are missing prices concentrated in a group?
The most expensive listing is not necessarily the best performer: price is not profit,
revenue, or booking demand.

In [91]:
inspection_cols = [
    'id', 'name', 'price', 'room_type', 'property_type',
    'neighbourhood_cleansed', 'accommodates', 'bedrooms', 'bathrooms', 'beds'
]
print('10 highest recorded prices')
display(priced.nlargest(10, 'price')[inspection_cols])
print('10 lowest positive recorded prices')
display(priced.nsmallest(10, 'price')[inspection_cols])
print('Largest bedroom counts — inspect whether these are comparable listing units')
display(df.nlargest(5, 'bedrooms')[inspection_cols])

10 highest recorded prices


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
20058,53056104,Madrid,36883.0,Entire home/apt,Entire rental unit,Universidad,6,3.0,1.5,NaN
162,34570478,Apartamento,22832.5,Entire home/apt,Entire loft,Sol,1,1.0,1.0,2.0
198,1496308350558748700,Plaza de la Villa doble con baño compartido,15000.0,Private room,Private room in home,Palacio,2,NaN,1.0,1.0
213,1482716640416035527,Centric 2 dormitorios,15000.0,Entire home/apt,Entire rental unit,Cortes,7,2.0,1.0,3.0
214,1496255451562926135,Centric 1 dormitorio,15000.0,Entire home/apt,Entire rental unit,Cortes,4,1.0,1.0,1.0
275,1498654933003051613,Buenavista apartamento,15000.0,Entire home/apt,Entire rental unit,Embajadores,4,1.0,1.0,1.0
6249,1496302830029735943,Espoz y Mina doble con baño compartido,15000.0,Private room,Private room in home,Cortes,2,NaN,1.0,1.0
6250,1496264019813624356,Núñez de Arce estudio,15000.0,Entire home/apt,Entire rental unit,Cortes,4,NaN,1.0,2.0
9465,1496323497168656654,Buenavista doble con baño compartido,15000.0,Private room,Private room in home,Embajadores,2,NaN,1.0,1.0
9466,1498653316318644494,Buenavista estudio,15000.0,Entire home/apt,Entire rental unit,Embajadores,3,1.0,1.0,1.0


10 lowest positive recorded prices


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
17765,42080666,MADRID CENTRO DESING,3.36,Private room,Private room in rental unit,San Isidro,1,NaN,NaN,NaN
22308,1507595,Flat 3Rooms Ventas up to 6 easy to park,4.65,Entire home/apt,Entire rental unit,Ventas,6,3.0,NaN,NaN
5378,1681014813315383176,13D7 Habitación cama doble con aire acondicionado,5.03,Private room,Private room in rental unit,Cortes,1,1.0,1.0,1.0
8524,880795197907807332,Encantador Duplex pleno centro,6.89,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
6397,650669100988899030,Apartamento diseño industrial en zona financiera,7.02,Entire home/apt,Entire rental unit,Cuatro Caminos,4,1.0,NaN,NaN
8525,611601130764966973,Precioso duplex en Madrid Centro,7.40,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
8522,861371947458329526,Acogedor Duplex en pleno Centro,7.46,Entire home/apt,Entire rental unit,Embajadores,3,1.0,NaN,NaN
1382,1220239709744410594,Habitacion en Carabanchel,7.71,Private room,Private room in rental unit,Abrantes,1,NaN,NaN,NaN
4736,1369528898864816023,Habitación en piso compartido,8.01,Private room,Private room in rental unit,Castillejos,1,NaN,NaN,NaN
5083,978252148209220621,Piso de 3 habitaciones dobles cerca del Bernabéu.,8.02,Entire home/apt,Entire rental unit,Colina,5,3.0,NaN,NaN


Largest bedroom counts — inspect whether these are comparable listing units


,id,name,price,room_type,property_type,neighbourhood_cleansed,accommodates,bedrooms,bathrooms,beds
18023,1705305290816860090,Beddy Hostels,47.2,Private room,Room in hotel,Simancas,1,50.0,1.0,1.0
8739,51204846,Habitación privada de 6,170.0,Private room,Private room in hostel,Embajadores,6,25.0,1.0,167.0
18575,1296653058099912706,Hostal Patria Doble baño compartido,NaN,Private room,Room in hotel,Sol,2,18.0,NaN,NaN
18576,1295882263524337884,Hostal Patria Habitación doble,NaN,Private room,Room in hotel,Sol,2,18.0,NaN,NaN
18577,1296662804005576431,Hostal Patria Individual baño compartido.,NaN,Private room,Room in hotel,Sol,1,18.0,NaN,NaN


In [92]:
# Check whether missing price data is concentrated in one source.
missing_by_source = df.assign(price_missing=df['price'].isna()).groupby('source').agg(
    listings=('id', 'size'),
    missing_prices=('price_missing', 'sum'),
    missing_price_fraction=('price_missing', 'mean')
)
missing_by_source['missing_price_percent'] = missing_by_source.pop('missing_price_fraction') * 100
display(missing_by_source.round(2))

,listings,missing_prices,missing_price_percent
source,,,
city scrape,18548,285,1.54
previous scrape,4160,3251,78.15


## Step 15 — Summarize findings and next questions

The cell below calculates a short summary from the data, so it updates when rerun.
Use the tables and plots to explain both the differences and their limitations.

In [93]:
print(f"Missing prices: {df['price'].isna().mean():.1%} of all listings.")
print(f"Median price: {priced['price'].median():.2f}; mean price: {priced['price'].mean():.2f}.")
print('Median prices and available-price counts by room type:')
display(price_summaries['room_type'][['priced_listings', 'median_price']].round(2))
print('Numerical variables most strongly associated with price by absolute Spearman correlation:')
price_relationships = correlation['price'].drop('price').dropna()
display(price_relationships.reindex(price_relationships.abs().sort_values(ascending=False).index).head(3).round(3))

Missing prices: 15.6% of all listings.
Median price: 125.00; mean price: 180.06.
Median prices and available-price counts by room type:


,priced_listings,median_price
room_type,,
Entire home/apt,13936,144.0
Hotel room,29,104.0
Private room,5065,64.0
Shared room,142,25.0


Numerical variables most strongly associated with price by absolute Spearman correlation:


accommodates    0.642
beds            0.549
bedrooms        0.462
Name: price, dtype: float64

**Questions to answer from your results:**
1. How different are mean and median price? What does the distribution explain?
2. Which numerical variables have the strongest association with price?
3. Which room types have higher median prices, and how many recorded prices support each result?
4. Among the common neighbourhoods and property types, where are median prices higher or lower?
5. Do neighbourhood differences remain within the same room type?
6. Which unusual listings should be checked against the original records?
7. Which groups have low price coverage, and how might this affect the conclusions?

**Suggested follow-up:** Verify unusual listing sizes and price quote definitions,
then compare listings of similar capacity within room type and neighbourhood.
For seasonality, collect repeated dated prices or booking data. These comparisons
describe associations in observed listing prices; they do not measure profitability
or prove what causes a higher price.

### Findings from this file

- Of 22,708 listings, 19,172 have recorded positive prices; 15.6% have missing prices.
- Median price is 125.00, below the mean of 180.06. The long upper tail makes the median a useful typical-price summary.
- Median prices are 144.00 for entire homes/apartments, 104.00 for hotel rooms, 64.00 for private rooms, and 25.00 for shared rooms. Only 29 hotel-room listings have prices, so interpret that median cautiously.
- Price has its strongest positive Spearman association among the selected variables with capacity (0.642), followed by beds (0.549) and bedrooms (0.462). Rating has only a weak association (0.082). These are unadjusted associations.
- Missing prices are concentrated in previous-scrape listings (78.15% missing), compared with city-scrape listings (1.54%). The price comparisons therefore mainly describe listings with prices in the city scrape.

These observations describe the supplied CSV. Rerun the summary cells if the data changes.

## Step 16 - Revenue, occupancy, and reviews: check the target first

**Business question:** Which characteristics are associated with higher annual revenue, and what could a new host investigate to improve bookings and guest experience?

Start here for the revenue investigation. This section reloads `listings.csv` into `revenue_data`, so it also runs independently of the earlier price EDA.

**The main finding:** This file's estimated revenue is constructed from price and estimated occupied nights. The occupied nights are in turn constructed from recent reviews and minimum stay length. These relationships describe the estimate's recipe; they do not establish that collecting an extra review causes the corresponding amount of real income.

The reconstruction below is inferred from this CSV and checked row by row. It is consistent with [Inside Airbnb's occupancy methodology](https://insideairbnb.com/data-assumptions/). The exact cap and rounding should be rechecked for a different export.

In [35]:
import json
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

TARGET = 'estimated_revenue_l365d'
REVIEW_TARGET = 'number_of_reviews_ltm'
revenue_data = pd.read_csv('listings.csv').drop_duplicates()
for col in ['price', TARGET]:
    revenue_data[col] = pd.to_numeric(
        revenue_data[col].astype('string').str.replace(r'[\$,]', '', regex=True),
        errors='coerce'
    )
for col in ['first_review', 'last_review', 'last_scraped']:
    revenue_data[col] = pd.to_datetime(revenue_data[col], errors='coerce')

# Keep unknown Superhost status missing; unknown is not the same as False.
revenue_data['superhost_binary'] = revenue_data['host_is_superhost'].map({'t': 1, 'f': 0})
revenue_data['superhost_status'] = revenue_data['host_is_superhost'].map(
    {'t': 'Superhost', 'f': 'Not Superhost'}
).fillna('Unknown')
revenue_data['days_since_first_review'] = (
    revenue_data['last_scraped'] - revenue_data['first_review']
).dt.days
revenue_data['host_experience_months'] = (
    12 * revenue_data['hosts_time_as_host_years']
    + revenue_data['hosts_time_as_host_months']
)

def amenity_count(value):
    if pd.isna(value):
        return np.nan
    try:
        items = json.loads(value)
        return len(set(items)) if isinstance(items, list) else np.nan
    except (TypeError, ValueError):
        return np.nan

revenue_data['amenity_count'] = revenue_data['amenities'].map(amenity_count)
# Missing or invalid revenue is excluded; genuine recorded zeros are retained.
revenue_data[TARGET] = revenue_data[TARGET].where(
    np.isfinite(revenue_data[TARGET]) & revenue_data[TARGET].ge(0)
)
revenue_data['log_revenue'] = np.log1p(revenue_data[TARGET])
revenue_observed = revenue_data.dropna(subset=[TARGET]).copy()

display(pd.Series({
    'all_listings': len(revenue_data),
    'duplicate_listing_ids': revenue_data['id'].duplicated().sum(),
    'known_revenue': len(revenue_observed),
    'missing_revenue': revenue_data[TARGET].isna().sum(),
    'recorded_zero_revenue': revenue_observed[TARGET].eq(0).sum(),
    'unknown_superhost_status': revenue_data['superhost_binary'].isna().sum(),
    'unique_hosts': revenue_data['host_id'].nunique()
}).to_frame('count'))
display(revenue_observed[TARGET].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2))

coverage = revenue_data.groupby('source', dropna=False).agg(
    listings=('id', 'size'), observed_revenue=(TARGET, 'count')
)
coverage['revenue_coverage_percent'] = 100 * coverage['observed_revenue'] / coverage['listings']
display(coverage.round(2))

,count
all_listings,22708
duplicate_listing_ids,0
known_revenue,19172
missing_revenue,3536
recorded_zero_revenue,4216
unknown_superhost_status,152
unique_hosts,8638


count      19172.0
mean      15966.17
std       26165.06
min            0.0
10%            0.0
25%          885.0
50%         8401.5
75%        22950.0
90%        39899.7
99%       99664.53
max      1427040.0
Name: estimated_revenue_l365d, dtype: Float64

,listings,observed_revenue,revenue_coverage_percent
source,,,
city scrape,18548,18263,98.46
previous scrape,4160,909,21.85


In [36]:
# Test the formula on rows with all required inputs. Do not fill missing inputs.
occupancy_inputs = ['number_of_reviews_ltm', 'minimum_nights', 'estimated_occupancy_l365d']
occupancy_check = revenue_data.dropna(subset=occupancy_inputs).copy()
occupancy_check['reconstructed_nights'] = (
    occupancy_check['number_of_reviews_ltm'] * 2
    * occupancy_check['minimum_nights'].clip(lower=3)
).clip(upper=255)
occupancy_match = np.isclose(
    occupancy_check['estimated_occupancy_l365d'], occupancy_check['reconstructed_nights']
)

revenue_check = revenue_data.dropna(subset=[TARGET, 'price', 'estimated_occupancy_l365d'])
revenue_error = revenue_check[TARGET] - (
    revenue_check['price'] * revenue_check['estimated_occupancy_l365d']
)
revenue_match = revenue_error.abs().le(0.500001)  # nearest whole currency unit
formula_results = pd.DataFrame({
    'check': ['Occupancy reconstruction', 'Price x occupied nights, within rounding'],
    'eligible_rows': [len(occupancy_check), len(revenue_check)],
    'matching_rows': [int(occupancy_match.sum()), int(revenue_match.sum())],
    'match_percent': [100 * occupancy_match.mean(), 100 * revenue_match.mean()]
})
display(formula_results.round(3))
print('Maximum absolute revenue rounding difference:', round(revenue_error.abs().max(), 4))
print('Rows excluded from occupancy reconstruction:', len(revenue_data) - len(occupancy_check))

Maximum absolute revenue rounding difference: 0.5
Rows excluded from occupancy reconstruction: 2


,check,eligible_rows,matching_rows,match_percent
0,Occupancy reconstruction,22706,22706,100.0
1,"Price x occupied nights, within rounding",19172,19172,100.0


### What this means

For the checked rows in this file:

```text
estimated bookings = reviews in the last 12 months x 2
assumed nights per booking = max(3, minimum_nights)
estimated occupied nights = min(255, estimated bookings x assumed nights per booking)
estimated revenue = price x estimated occupied nights, rounded to a whole unit
```

- `estimated_occupancy_l365d` is a **number of nights**, not a percentage. Divide by 365 for an annual fraction; this does not measure occupancy of the nights a host actually offered.
- `number_of_reviews_ltm` covers the same broad annual window as the target. `number_of_reviews` is lifetime reviews; `reviews_per_month` is a lifetime monthly average, not a guest review probability. See the [data dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit).
- An estimated zero can mean no recent reviews, even if the property had unreviewed stays. Missing revenue remains missing.
- Increasing price or minimum nights mechanically increases this estimate with the other inputs fixed. Real bookings can respond differently, so this recipe cannot find the price or stay rule that maximizes actual revenue.
- The price is a scraped quote, not the average rate actually collected over the last year. The dictionary describes local currency; this file's nonmissing quote currency codes are EUR. Removing `$` does not convert currencies.

Keep occupancy, review proxies, and formula inputs visible for this audit, but do not use them as independent evidence of a profitable intervention. A model for a **new listing** should use information available before launch; a forecast for an existing listing can use earlier reviews to predict a later, independently observed revenue period.

## Step 17 - Numerical versus numerical relationships

Use **Spearman** for an initial ranking of these skewed variables, and compare with **Pearson** for linear relationships. Each row reports the number of listings with both values. A correlation near zero does not rule out a threshold or another nonlinear relationship.

Split formula inputs and review proxies from property and host characteristics. This prevents the strongest correlation from being mistaken for the best action to take.

In [37]:
formula_and_review_features = [
    'estimated_occupancy_l365d', 'number_of_reviews_ltm', 'number_of_reviews',
    'reviews_per_month', 'number_of_reviews_l30d', 'price', 'minimum_nights'
]
property_host_features = [
    'accommodates', 'bedrooms', 'bathrooms', 'beds', 'amenity_count',
    'host_experience_months', 'calculated_host_listings_count',
    'availability_365', 'days_since_first_review', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_accuracy', 'review_scores_communication',
    'review_scores_checkin', 'review_scores_location', 'review_scores_value'
]

def numeric_associations(data, target, features):
    rows = []
    for feature in features:
        pair = data[[feature, target]].replace([np.inf, -np.inf], np.nan).dropna()
        sufficient = len(pair) >= 3 and pair[feature].nunique() > 1 and pair[target].nunique() > 1
        rows.append({
            'feature': feature, 'n': len(pair),
            'spearman': pair.corr(method='spearman').iloc[0, 1] if sufficient else np.nan,
            'pearson': pair.corr(method='pearson').iloc[0, 1] if sufficient else np.nan
        })
    table = pd.DataFrame(rows).set_index('feature')
    return table.loc[table['spearman'].abs().sort_values(ascending=False).index]

formula_associations = numeric_associations(revenue_data, TARGET, formula_and_review_features)
property_associations = numeric_associations(revenue_data, TARGET, property_host_features)
print('Formula inputs and review proxies: interpret as construction/dependence checks')
display(formula_associations.round(3))
print('Other listing characteristics: unadjusted associations, not causal effects')
display(property_associations.round(3))

px.bar(
    property_associations.sort_values('spearman').reset_index(),
    x='spearman', y='feature', orientation='h', hover_data=['n', 'pearson'],
    range_x=[-1, 1], height=650,
    title='Listing characteristics and estimated revenue: Spearman association'
).show()

Formula inputs and review proxies: interpret as construction/dependence checks
Other listing characteristics: unadjusted associations, not causal effects


,n,spearman,pearson
feature,,,
estimated_occupancy_l365d,19172,0.909,0.559
number_of_reviews_ltm,19172,0.892,0.422
number_of_reviews,19172,0.771,0.343
reviews_per_month,16464,0.682,0.296
number_of_reviews_l30d,19172,0.671,0.310
price,19172,0.355,0.144
minimum_nights,19171,-0.235,-0.149


,n,spearman,pearson
feature,,,
amenity_count,19172,0.392,0.266
accommodates,19172,0.327,0.329
beds,17888,0.256,0.163
bedrooms,14840,0.163,0.192
days_since_first_review,16464,0.154,0.098
host_experience_months,19040,0.146,0.093
availability_365,19172,-0.123,-0.047
review_scores_value,16463,0.119,0.152
bathrooms,17029,0.113,0.174


In [38]:
# Only the scatter charts are sampled. All tables use every eligible listing.
review_scatter = revenue_data.sample(n=min(2500, len(revenue_data)), random_state=42)
px.scatter(
    review_scatter, x=REVIEW_TARGET, y='estimated_occupancy_l365d',
    color='room_type', opacity=.4, hover_data=['minimum_nights'],
    title='Reviews and estimated occupied nights: formula and cap are visible'
).show()
px.scatter(
    review_scatter, x=REVIEW_TARGET, y='log_revenue', color='room_type', opacity=.4,
    hover_data=[TARGET, 'price'],
    title='Recent reviews and log(1 + estimated revenue): zeros are retained'
).show()

## Step 18 - What makes a listing receive reviews?

Separate **review volume** from **the probability a guest reviews**:

```text
expected review count approximately equals
completed stays x probability that a stay receives a review
```

For example, 20 stays with a 50% review rate and 10 stays with a 100% review rate both produce 10 reviews. Counts alone cannot tell them apart. At the same number of occupied nights, shorter stays also create more review opportunities.

Use recent review count as a **review activity** outcome here. Actual completed stays are absent, so this file cannot identify review probability. Do not divide reviews by bookings reconstructed from those same reviews; that would simply recover the model's assumed review rate.

Investigate property size, amenities, location, minimum stay, host experience, and guest ratings as hypotheses. Price, demand, time on the market, and guest mix can affect several of these variables together. Current ratings and status can also be consequences of earlier stays.

In [39]:
review_features = [
    'price', 'minimum_nights', 'accommodates', 'bedrooms', 'bathrooms',
    'amenity_count', 'host_experience_months', 'calculated_host_listings_count',
    'days_since_first_review', 'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_accuracy', 'review_scores_communication', 'review_scores_checkin',
    'review_scores_location', 'review_scores_value'
]
review_associations = numeric_associations(revenue_data, REVIEW_TARGET, review_features)

# Sensitivity analysis: at least a year since the first recorded review.
# This is NOT a known launch date or proof the listing operated all year.
established = revenue_data[revenue_data['days_since_first_review'].ge(365)].copy()
established_review_associations = numeric_associations(established, REVIEW_TARGET, review_features)
review_comparison = review_associations[['n', 'spearman']].join(
    established_review_associations[['n', 'spearman']], rsuffix='_established'
)
display(review_comparison.round(3))
print('All listings:', len(revenue_data), '| First review at least 365 days ago:', len(established))
print('The established subset excludes listings without a first review and does not measure active days.')

candidate_operational_fields = [
    'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'instant_bookable'
]
print('Coverage of additional hosting variables:')
display(revenue_data[candidate_operational_fields].notna().sum().to_frame('nonmissing_listings'))

All listings: 22708 | First review at least 365 days ago: 14799
The established subset excludes listings without a first review and does not measure active days.
Coverage of additional hosting variables:


,n,spearman,n_established,spearman_established
feature,,,,
amenity_count,22708,0.375,14799,0.276
minimum_nights,22706,-0.343,14797,-0.330
accommodates,22708,0.169,14799,0.162
review_scores_checkin,18992,-0.133,14799,-0.096
review_scores_location,18991,-0.122,14798,-0.076
price,19172,0.105,12464,0.079
review_scores_communication,18992,-0.088,14799,-0.049
review_scores_accuracy,18992,-0.071,14799,-0.031
review_scores_value,18990,0.038,14797,0.066


,nonmissing_listings
host_response_time,0
host_response_rate,0
host_acceptance_rate,0
instant_bookable,0


### Why weak rating correlations do not mean negative reviews are harmless

This file contains listing-level average scores, not separate positive/negative review counts or review text. It cannot directly compare the effects of positive and negative reviews.

Average ratings also have a compressed range, and new listings may have very few reviews. Missing scores are not low scores. Show rating bands and a sensitivity analysis requiring at least 10 lifetime reviews; this threshold is an explicit analysis choice, not an Airbnb rule. Neither comparison estimates the future effect of a bad review.

In [40]:
rating_data = revenue_data.copy()
rating_data['rating_band'] = pd.cut(
    rating_data['review_scores_rating'], bins=[0, 4, 4.5, 4.8, 5.01],
    labels=['Below 4.0', '4.0 to <4.5', '4.5 to <4.8', '4.8 to 5.0'], right=False
)
rating_data['rating_band'] = rating_data['rating_band'].cat.add_categories('No rating').fillna('No rating')
rating_summary = rating_data.groupby('rating_band', observed=True).agg(
    listings=('id', 'size'), observed_revenue=(TARGET, 'count'),
    median_revenue=(TARGET, 'median'), median_recent_reviews=(REVIEW_TARGET, 'median')
)
display(rating_summary.round(2))
display(revenue_data['review_scores_rating'].describe(percentiles=[.1, .25, .5, .75, .9]).round(2))

rating_sensitivity = []
for name, subset in [
    ('All with a rating', revenue_data),
    ('At least 10 lifetime reviews', revenue_data[revenue_data['number_of_reviews'].ge(10)])
]:
    row = numeric_associations(subset, TARGET, ['review_scores_rating']).iloc[0]
    rating_sensitivity.append({'sample': name, **row.to_dict()})
display(pd.DataFrame(rating_sensitivity).round(3))

,listings,observed_revenue,median_revenue,median_recent_reviews
rating_band,,,,
Below 4.0,837,722,1536.0,1.0
4.0 to <4.5,3004,2578,7344.0,6.0
4.5 to <4.8,7003,6289,15846.0,16.0
4.8 to 5.0,8149,6875,11781.0,7.0
No rating,3715,2708,0.0,0.0


count    18993.00
mean         4.64
std          0.48
min          1.00
10%          4.24
25%          4.54
50%          4.75
75%          4.90
90%          5.00
max          5.00
Name: review_scores_rating, dtype: float64

,sample,n,spearman,pearson
0,All with a rating,16464.0,0.066,0.140
1,At least 10 lifetime reviews,11949.0,0.184,0.157


## Step 19 - Numerical versus categorical: how to measure it

| Variables | Use | How to read it |
|---|---|---|
| Numeric + numeric | Spearman / Pearson | Signed association from -1 to +1 |
| Binary category + numeric | Point-biserial correlation | Pearson after meaningful 0/1 coding; sign depends on the coding |
| Unordered category with 3+ groups + numeric | Group summaries, box plots, eta-squared | Eta-squared is the share of observed variation represented by differences in group means |

Do **not** encode neighbourhoods as 1, 2, 3 and correlate those numbers with revenue. The numbering has no meaningful order or distance. One-hot encoding is appropriate when you later fit a multivariable model.

**Eta-squared** = between-group sum of squares / total sum of squares. It ranges from 0 to 1 and has no positive/negative direction. For example, 0.06 means that group-mean differences account for 6% of variation in this sample, not that changing category would increase revenue by 6%. The correlation ratio eta is its square root. See [ANOVA and effect-size documentation](https://pingouin-stats.org/generated/pingouin.anova.html).

Calculate on raw revenue and on `log(1 + revenue)`, which compresses the long upper tail and retains zeros. These are two different scales; log-scale results do not describe variation in raw currency amounts. Large numbers of categories can inflate in-sample eta-squared, so category rankings are descriptive and should not be read as a ranking of causal importance.

In [41]:
revenue_categories = ['neighbourhood_cleansed', 'room_type', 'property_type', 'superhost_status']
revenue_group_summaries = {}
for category in revenue_categories:
    summary = revenue_data.groupby(category, dropna=False, observed=True).agg(
        listings=('id', 'size'), known_revenue=(TARGET, 'count'),
        median_revenue=(TARGET, 'median'), mean_revenue=(TARGET, 'mean'),
        revenue_p25=(TARGET, lambda x: x.quantile(.25)),
        revenue_p75=(TARGET, lambda x: x.quantile(.75)),
        median_recent_reviews=(REVIEW_TARGET, 'median')
    )
    summary['revenue_coverage_percent'] = 100 * summary['known_revenue'] / summary['listings']
    summary['small_revenue_sample'] = summary['known_revenue'].lt(30)
    revenue_group_summaries[category] = summary.sort_values('median_revenue', ascending=False)
    print(category, '- full table available in revenue_group_summaries[category]')
    # Display up to 15 largest groups; all groups remain in the saved table.
    display(summary.nlargest(15, 'listings').sort_values('median_revenue', ascending=False).round(2))

    common = summary.nlargest(12, 'listings').index
    chart_data = revenue_observed[revenue_observed[category].isin(common)]
    order = summary.loc[common].sort_values('median_revenue', ascending=False).index.tolist()
    px.box(
        chart_data, x=category, y='log_revenue', points=False,
        category_orders={category: order}, height=500,
        title=f'Log(1 + estimated revenue) by {category}: up to 12 largest groups'
    ).show()

neighbourhood_cleansed - full table available in revenue_group_summaries[category]
room_type - full table available in revenue_group_summaries[category]
property_type - full table available in revenue_group_summaries[category]
superhost_status - full table available in revenue_group_summaries[category]


,listings,known_revenue,median_revenue,mean_revenue,revenue_p25,revenue_p75,median_recent_reviews,revenue_coverage_percent,small_revenue_sample
neighbourhood_cleansed,,,,,,,,,
Sol,1240,1125,15570.0,27889.21,2112.0,37152.0,11.0,90.73,False
Cortes,1030,904,15095.5,21885.46,2595.0,34157.25,9.0,87.77,False
Palos de Moguer,369,305,14025.0,18254.11,2808.0,27318.0,13.0,82.66,False
Palacio,1681,1469,13416.0,21494.97,2394.0,31875.0,8.0,87.39,False
Universidad,1946,1658,13292.0,20041.27,2231.25,28792.5,8.0,85.2,False
Recoletos,307,264,13281.0,26877.81,0.0,34867.5,4.0,85.99,False
Embajadores,2453,2048,11449.5,18429.53,2100.0,26465.5,7.0,83.49,False
Justicia,1120,943,11242.0,19336.72,1440.5,29394.0,6.0,84.2,False
Goya,367,294,9314.5,17802.87,0.0,25282.75,2.0,80.11,False


,listings,known_revenue,median_revenue,mean_revenue,revenue_p25,revenue_p75,median_recent_reviews,revenue_coverage_percent,small_revenue_sample
room_type,,,,,,,,,
Entire home/apt,15899,13936,12606.5,19823.12,2488.0,28560.0,7.0,87.65,False
Private room,6611,5065,2208.0,5813.03,0.0,9366.0,1.0,76.61,False
Shared room,159,142,1200.0,2321.03,127.5,3705.75,5.0,89.31,False
Hotel room,39,29,594.0,2611.0,0.0,3120.0,1.0,74.36,True


,listings,known_revenue,median_revenue,mean_revenue,revenue_p25,revenue_p75,median_recent_reviews,revenue_coverage_percent,small_revenue_sample
property_type,,,,,,,,,
Entire loft,384,326,20587.0,22091.9,7428.0,32108.5,16.5,84.9,False
Entire condo,433,367,15861.0,20482.28,0.0,30704.0,6.0,84.76,False
Entire rental unit,14291,12594,12349.0,19610.67,2466.0,28368.0,7.0,88.13,False
Entire home,291,223,9915.0,19837.74,1410.0,30116.5,3.0,76.63,False
Entire serviced apartment,309,269,8640.0,26816.56,1550.0,29268.0,4.0,87.06,False
Private room in condo,235,166,8318.0,9167.64,2092.5,14025.0,7.0,70.64,False
Room in aparthotel,78,76,6522.0,11405.04,2215.5,20458.5,10.5,97.44,False
Private room in hostel,327,318,3026.5,7444.25,0.0,11205.75,4.0,97.25,False
Private room in bed and breakfast,118,96,2961.0,6484.93,0.0,10296.0,2.5,81.36,False


,listings,known_revenue,median_revenue,mean_revenue,revenue_p25,revenue_p75,median_recent_reviews,revenue_coverage_percent,small_revenue_sample
superhost_status,,,,,,,,,
Superhost,5487,5151,18916.0,25369.77,8129.0,34125.0,21.0,93.88,False
Not Superhost,17069,13889,5334.0,12585.14,0.0,18041.0,2.0,81.37,False
Unknown,152,132,1085.5,4762.33,0.0,5963.25,1.0,86.84,False


In [42]:
def eta_squared(data, category, outcome):
    pair = data[[category, outcome]].dropna()
    if len(pair) < 2 or pair[category].nunique() < 2:
        return np.nan
    values = pair[outcome]
    ss_total = ((values - values.mean()) ** 2).sum()
    if ss_total == 0:
        return np.nan
    group_means = pair.groupby(category, observed=True)[outcome].transform('mean')
    ss_between = ((group_means - values.mean()) ** 2).sum()
    return float(ss_between / ss_total)

association_rows = []
for category in ['neighbourhood_cleansed', 'room_type', 'property_type', 'host_is_superhost']:
    pair = revenue_data.dropna(subset=[category, TARGET])
    group_counts = pair[category].value_counts()
    association_rows.append({
        'category': category, 'n': len(pair), 'groups': len(group_counts),
        'groups_below_30': group_counts.lt(30).sum(),
        'eta_squared_raw_revenue': eta_squared(pair, category, TARGET),
        'eta_squared_log_revenue': eta_squared(pair, category, 'log_revenue'),
        'eta_squared_recent_reviews': eta_squared(revenue_data, category, REVIEW_TARGET),
        'n_for_recent_reviews': revenue_data[[category, REVIEW_TARGET]].dropna().shape[0]
    })
category_associations = pd.DataFrame(association_rows).set_index('category')
display(category_associations.round(4))

,n,groups,groups_below_30,eta_squared_raw_revenue,eta_squared_log_revenue,eta_squared_recent_reviews,n_for_recent_reviews
category,,,,,,,
neighbourhood_cleansed,19172,127,32,0.0484,0.0416,0.0416,22708
room_type,19172,4,1,0.0580,0.0602,0.0088,22708
property_type,19172,55,36,0.0605,0.0688,0.0261,22708
host_is_superhost,19040,2,0,0.0469,0.0877,0.0834,22556


**Association strength and a statistical test answer different questions.** Eta-squared summarizes group separation. [Welch ANOVA](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html) tests equality of means while allowing unequal variances. [Kruskal-Wallis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html) is a rank-based comparison; interpret it as a median comparison only when distribution shapes are comparable. A small p-value does not establish a large or useful difference, or say which groups differ.

This notebook reports descriptive effect sizes. Multiple listings share hosts, so standard tests treating every listing as independent can understate uncertainty. For inference, use host-clustered resampling or an appropriate clustered model, and adjust for multiple comparisons when testing many groups. No p-values are needed to read the EDA tables.

## Step 20 - Superhost versus non-Superhost

`host_is_superhost` is binary: code `t` as 1 and `f` as 0. The [point-biserial correlation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pointbiserialr.html) is exactly Pearson's correlation with that 0/1 variable. We can calculate the coefficient with pandas; SciPy is not needed for this descriptive calculation.

With this coding, a positive coefficient means the Superhost group has a higher mean outcome. Also inspect medians, spreads, missing-data coverage, and sample sizes.

**Selection matters:** [Superhost qualification](https://www.airbnb.com/help/article/829) already depends on completed stays, ratings, responsiveness, and cancellations, and is evaluated across a host's listings. Current status versus the last year's estimated revenue mixes past performance with any possible badge effect. The gap below is an association, not an estimate of the revenue gained by becoming a Superhost.

In [43]:
display(revenue_group_summaries['superhost_status'].round(2))
superhost_rows = []
for outcome in [TARGET, 'log_revenue', REVIEW_TARGET]:
    pair = revenue_data[['superhost_binary', outcome]].dropna()
    coefficient = pair.corr(method='pearson').iloc[0, 1] if pair['superhost_binary'].nunique() == 2 else np.nan
    superhost_rows.append({'outcome': outcome, 'n': len(pair), 'point_biserial_r': coefficient})
superhost_associations = pd.DataFrame(superhost_rows).set_index('outcome')
display(superhost_associations.round(4))

,listings,known_revenue,median_revenue,mean_revenue,revenue_p25,revenue_p75,median_recent_reviews,revenue_coverage_percent,small_revenue_sample
superhost_status,,,,,,,,,
Superhost,5487,5151,18916.0,25369.77,8129.0,34125.0,21.0,93.88,False
Not Superhost,17069,13889,5334.0,12585.14,0.0,18041.0,2.0,81.37,False
Unknown,152,132,1085.5,4762.33,0.0,5963.25,1.0,86.84,False


,n,point_biserial_r
outcome,,
estimated_revenue_l365d,19040,0.2165
log_revenue,19040,0.2962
number_of_reviews_ltm,22556,0.2887


## Step 21 - Compare more similar listings

Start by holding **neighbourhood, room type, and guest capacity** constant. Compare Superhost and non-Superhost medians only in cells with at least 20 observed revenues per group. This is a practical stability rule; it is not a significance test.

This comparison still leaves differences in bedrooms, bathrooms, condition, property type, pricing, tenure, amenities, and management. A controlled model can add those variables, but a snapshot still cannot isolate a causal badge effect. Room type and property type overlap, so avoid interpreting their model coefficients as independent without checking redundancy.

In [44]:
peer_keys = ['neighbourhood_cleansed', 'room_type', 'accommodates']
peer_data = revenue_data.dropna(subset=peer_keys + ['superhost_binary', TARGET])
peer_medians = peer_data.groupby(peer_keys + ['superhost_binary'])[TARGET].median().unstack('superhost_binary')
peer_counts = peer_data.groupby(peer_keys + ['superhost_binary'])[TARGET].size().unstack('superhost_binary')
peer_medians = peer_medians.reindex(columns=[0.0, 1.0])
peer_counts = peer_counts.reindex(columns=[0.0, 1.0]).fillna(0)
enough_data = peer_counts.ge(20).all(axis=1)
superhost_peer_comparison = pd.DataFrame({
    'not_superhost_n': peer_counts.loc[enough_data, 0.0],
    'superhost_n': peer_counts.loc[enough_data, 1.0],
    'not_superhost_median': peer_medians.loc[enough_data, 0.0],
    'superhost_median': peer_medians.loc[enough_data, 1.0]
})
superhost_peer_comparison['median_gap'] = (
    superhost_peer_comparison['superhost_median'] - superhost_peer_comparison['not_superhost_median']
)
superhost_peer_comparison['total_n'] = (
    superhost_peer_comparison['superhost_n'] + superhost_peer_comparison['not_superhost_n']
)
print('Eligible comparison cells:', len(superhost_peer_comparison))
print('Listings in eligible cells:', int(superhost_peer_comparison['total_n'].sum()))
display(superhost_peer_comparison.nlargest(15, 'total_n').round(2))
print('Full table: superhost_peer_comparison. These are descriptive within-group differences.')

Eligible comparison cells: 41
Listings in eligible cells: 7259
Full table: superhost_peer_comparison. These are descriptive within-group differences.


not_superhost_n  ...  total_n
neighbourhood_cleansed room_type       accommodates                   ...         
Embajadores            Entire home/apt 2                         381  ...      549
                                       4                         357  ...      466
Palacio                Entire home/apt 4                         314  ...      463
Universidad            Entire home/apt 2                         320  ...      436
                                       4                         285  ...      404
Palacio                Entire home/apt 2                         238  ...      340
Sol                    Entire home/apt 4                         204  ...      292
Justicia               Entire home/apt 4                         174  ...      246
                                       2                         158  ...      243
Cortes                 Entire home/apt 4                         172  ...      236
Sol                    Entire home/apt 2                         150  ...      225
Embajadores            Private room    2                         151  ...      211
Universidad            Private room    2                         161  ...      205
Embajadores            Entire home/apt 3                         146  ...      203
Cortes                 Entire home/apt 2                         128  ...      201

[15 rows x 6 columns]

In [45]:
# Compare neighbourhoods within room type. Small cells are masked, not set to zero.
largest_neighbourhoods = revenue_data['neighbourhood_cleansed'].value_counts().head(12).index
neighbourhood_sample = revenue_observed[
    revenue_observed['neighbourhood_cleansed'].isin(largest_neighbourhoods)
]
neighbourhood_room_median = neighbourhood_sample.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values=TARGET, aggfunc='median'
)
neighbourhood_room_count = neighbourhood_sample.pivot_table(
    index='neighbourhood_cleansed', columns='room_type', values=TARGET, aggfunc='count', fill_value=0
)
display(neighbourhood_room_count)
display(neighbourhood_room_median.where(neighbourhood_room_count.ge(30)).round(2))

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,144,0,103,3
Cortes,718,0,186,0
Cuatro Caminos,257,0,85,54
Embajadores,1630,8,387,23
Goya,257,0,37,0
Guindalera,289,0,43,12
Justicia,742,0,200,1
Palacio,1197,1,260,11
Palos de Moguer,227,0,75,3


room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Argüelles,13366.5,<NA>,0.0,<NA>
Cortes,20885.5,<NA>,1050.0,<NA>
Cuatro Caminos,8934.0,<NA>,1032.0,450.0
Embajadores,14341.0,<NA>,4656.0,<NA>
Goya,10472.0,<NA>,0.0,<NA>
Guindalera,7903.0,<NA>,1142.0,<NA>
Justicia,14616.0,<NA>,4768.0,<NA>
Palacio,17611.0,<NA>,3702.0,<NA>
Palos de Moguer,21930.0,<NA>,3384.0,<NA>


## Step 22 - Turn the EDA into a hosting investigation

### What is supported by this file

- The revenue/review relationship is partly mechanical; the estimator uses recent reviews regardless of their score.
- Category and Superhost comparisons identify peer groups and hypotheses. They do not identify a revenue-maximizing intervention.
- All four requested categorical factors are compared separately. The numerical analysis includes capacity, bedrooms, beds, bathrooms, amenities count, host experience, and review sub-scores.
- Host response time, response rate, acceptance rate, and Instant Book are entirely missing in this export. Their associations cannot be estimated here.
- An amenities count measures listing-description breadth, not quality or the value of a particular amenity.

### What to do as a host, and how to measure it

| Question | Practical action or next analysis | Measure using your own operations |
|---|---|---|
| Can I earn positive feedback? | Maintain cleanliness, accurate photos/details, clear arrival instructions, and timely communication; fix recurring complaints | Overall score, category scores, and recurring review themes |
| Are guests leaving reviews? | Send a brief, neutral request for an honest review after checkout and review guests promptly | Reviews / eligible completed stays in the same period |
| Is the price helping revenue? | Compare similar local listings and test rates while tracking conversion and seasonality | Actual revenue, occupied nights, and average realized nightly rate |
| Should I change minimum stay? | Compare short-stay demand with turnover costs and longer bookings | Revenue and contribution after variable costs per available night |
| Does Superhost status predict future performance? | Collect repeated snapshots around status changes and compare similar hosts with stable status | Subsequent actual bookings/revenue; account for prior trends and seasonality |

Airbnb describes [ratings and guest feedback](https://www.airbnb.com/resources/hosting-homes/a/why-reviews-matter-41) as signals for guest choice and improvement. These service actions are hypotheses to monitor, not effects measured by this CSV. Guests normally have [14 days after checkout](https://www.airbnb.com/help/article/995) to review a home stay. Request honest feedback without incentives or pressure, consistent with [Airbnb's review policy](https://www.airbnb.com/help/article/2673).

For a true review-rate investigation, collect listing ID, stay/checkout date, eligible completed stays, review received/date, score, and actual paid revenue. Compare matching stay cohorts after the review window has closed. Review text or individual scores are needed to distinguish positive and negative experiences; listing averages cannot reconstruct them.

For a future revenue model, predict a later period's actual revenue using earlier property/host information. Encode nominal categories with dummy variables, account for repeated hosts, and evaluate on later dates with hosts separated where possible. Do not use review-derived estimated occupancy or same-period reviews to claim independent predictive evidence for this constructed target. Avoid controlling for occupancy when trying to estimate a total effect that could work through bookings.

In [46]:
# Optional: enter YOUR listing characteristics to select peers.
# None means no filter. Results are market benchmarks, not forecasts for a new listing.
my_property = {
    'neighbourhood_cleansed': None,  # exact spelling from the category tables
    'room_type': None,               # e.g. 'Entire home/apt'
    'property_type': None,           # e.g. 'Entire rental unit'
    'accommodates': None
}
my_peers = revenue_data.copy()
for field, value in my_property.items():
    if value is not None:
        my_peers = my_peers[my_peers[field].eq(value)]
if all(value is None for value in my_property.values()):
    print('No property filters entered: this is the full market, not a tailored peer group.')
print('Peer listings:', len(my_peers), '| Known revenue:', my_peers[TARGET].notna().sum())
if my_peers[TARGET].notna().any():
    display(my_peers[[TARGET, 'price', REVIEW_TARGET]].describe(percentiles=[.25, .5, .75]).round(2))
else:
    print('No observed revenues for these filters. Check spelling or broaden the peer group.')

No property filters entered: this is the full market, not a tailored peer group.
Peer listings: 22708 | Known revenue: 19172


,estimated_revenue_l365d,price,number_of_reviews_ltm
count,19172.0,19172.0,22708.00
mean,15966.17,180.06,16.38
std,26165.06,555.62,25.27
min,0.0,3.36,0.00
25%,885.0,83.74,0.00
50%,8401.5,125.0,5.00
75%,22950.0,184.0,25.00
max,1427040.0,36883.0,819.00


## Step 23 - Screen possible purchases across the market

**Your scenario:** You can choose any neighbourhood, property type, room type, and capacity. Rank **revenue per listing**, not the total revenue of a neighbourhood (which mostly rewards places with more listings).

The first table ranks neighbourhood + room type + property type combinations. The second also separates guest capacity into 1-2, 3-4, 5-6, and 7+ guests. All recorded zeros stay in the comparisons. We require at least **30 recorded revenues and 10 distinct hosts with recorded revenue** per group, so one unusual listing or one large operator is less likely to dominate a candidate. These thresholds are transparent screening choices, not guarantees of reliability.

Read the median alongside the 25th and 75th percentiles, revenue coverage, and host counts. The 25th percentile is an observed lower quartile, not a forecast or confidence bound. Very broad capacity groups can still mix different properties.

We also rank currently non-Superhost listings as a sensitivity check for a host who has not yet earned that status. Existing non-Superhosts are not necessarily new hosts, so this is only a starting benchmark.

**This screens candidates for higher estimated gross revenue.** Purchase price, financing, renovation, cleaning, maintenance, fees, and time offered are absent, so the tables cannot identify the best investment return or an optimal purchase budget. This snapshot also cannot support an exact optimal nightly price. Check the candidate property and collect costs and actual local booking evidence before treating a segment median as a forecast.

In [47]:
purchase_data = revenue_data.copy()
purchase_data['capacity_band'] = pd.cut(
    purchase_data['accommodates'], bins=[0, 2, 4, 6, np.inf],
    labels=['1-2 guests', '3-4 guests', '5-6 guests', '7+ guests']
)

def rank_purchase_segments(data, keys, minimum_revenues=30, minimum_hosts=10):
    summary = data.groupby(keys, observed=True, dropna=True).agg(
        listings=('id', 'size'), known_revenue=(TARGET, 'count'),
        median_revenue=(TARGET, 'median'),
        revenue_p25=(TARGET, lambda x: x.quantile(.25)),
        revenue_p75=(TARGET, lambda x: x.quantile(.75)),
        median_price=('price', 'median'), median_capacity=('accommodates', 'median'),
        median_recent_reviews=(REVIEW_TARGET, 'median'),
        superhost_share_known=('superhost_binary', 'mean')
    )
    known = data.dropna(subset=[TARGET])
    summary['hosts_with_revenue'] = known.groupby(keys, observed=True)['host_id'].nunique()
    summary['revenue_coverage_percent'] = 100 * summary['known_revenue'] / summary['listings']
    enough = summary['known_revenue'].ge(minimum_revenues) & summary['hosts_with_revenue'].ge(minimum_hosts)
    return summary.loc[enough].sort_values(['median_revenue', 'known_revenue'], ascending=False)

purchase_keys = ['neighbourhood_cleansed', 'room_type', 'property_type']
purchase_segments = rank_purchase_segments(purchase_data, purchase_keys)
purchase_segments_by_capacity = rank_purchase_segments(purchase_data, purchase_keys + ['capacity_band'])
purchase_segments_non_superhost = rank_purchase_segments(
    purchase_data[purchase_data['superhost_binary'].eq(0)], purchase_keys + ['capacity_band']
)
for title, table in [
    ('Neighbourhood + room type + property type', purchase_segments),
    ('Also grouped by guest capacity', purchase_segments_by_capacity),
    ('Non-Superhost listings, also grouped by capacity', purchase_segments_non_superhost)
]:
    print(title, '| eligible groups:', len(table))
    display(table.head(15).round(2))

if not purchase_segments_by_capacity.empty:
    chart = purchase_segments_by_capacity.head(12).reset_index()
    chart['segment'] = chart[purchase_keys + ['capacity_band']].astype(str).agg(' | '.join, axis=1)
    chart['below_median'] = chart['median_revenue'] - chart['revenue_p25']
    chart['above_median'] = chart['revenue_p75'] - chart['median_revenue']
    px.bar(
        chart.iloc[::-1], x='median_revenue', y='segment', orientation='h',
        error_x='above_median', error_x_minus='below_median',
        hover_data=['known_revenue', 'hosts_with_revenue', 'revenue_coverage_percent'],
        height=650,
        title='Candidate segments: median estimated revenue, bars show the middle 50%'
    ).show()
print('The complete rankings remain available in purchase_segments, purchase_segments_by_capacity, and purchase_segments_non_superhost.')
print('Repeat the screening on a later snapshot before relying on the ranking.')

Neighbourhood + room type + property type | eligible groups: 111
Also grouped by guest capacity | eligible groups: 132
Non-Superhost listings, also grouped by capacity | eligible groups: 98
The complete rankings remain available in purchase_segments, purchase_segments_by_capacity, and purchase_segments_non_superhost.
Repeat the screening on a later snapshot before relying on the ranking.


,,,listings,known_revenue,median_revenue,revenue_p25,revenue_p75,median_price,median_capacity,median_recent_reviews,superhost_share_known,hosts_with_revenue,revenue_coverage_percent
neighbourhood_cleansed,room_type,property_type,,,,,,,,,,,
Universidad,Entire home/apt,Entire condo,44,38,38299.0,15769.5,45900.0,160.0,4.0,25.0,0.61,19.0,86.36
Embajadores,Entire home/apt,Entire serviced apartment,37,37,29101.0,5160.0,44087.0,203.0,4.0,31.0,0.54,13.0,100.0
Sol,Entire home/apt,Entire rental unit,819,759,25740.0,7758.0,45249.0,186.0,4.0,19.0,0.30,376.0,92.67
Embajadores,Entire home/apt,Entire loft,72,64,22631.5,7731.0,35056.5,136.97,2.5,31.5,0.43,43.0,88.89
Palacio,Entire home/apt,Entire loft,38,31,21648.0,9600.0,34042.5,132.0,3.0,26.0,0.53,22.0,81.58
Cortes,Entire home/apt,Entire rental unit,740,667,21318.0,6050.0,38250.0,168.0,4.0,15.0,0.31,312.0,90.14
Palos de Moguer,Entire home/apt,Entire rental unit,220,191,21082.0,7374.0,28815.5,135.0,4.0,17.0,0.32,126.0,86.82
Embajadores,Entire home/apt,Entire condo,65,52,19725.0,0.0,28335.0,112.77,3.0,6.0,0.31,32.0,80.0
Jerónimos,Entire home/apt,Entire rental unit,59,54,19195.0,0.0,36731.25,189.32,4.0,4.0,0.24,36.0,91.53


,,,,listings,known_revenue,median_revenue,revenue_p25,revenue_p75,median_price,median_capacity,median_recent_reviews,superhost_share_known,hosts_with_revenue,revenue_coverage_percent
neighbourhood_cleansed,room_type,property_type,capacity_band,,,,,,,,,,,
Sol,Entire home/apt,Entire rental unit,7+ guests,77,71,69150.0,27108.0,97707.0,397.0,8.0,25.0,0.31,54.0,92.21
Cortes,Entire home/apt,Entire rental unit,7+ guests,59,55,55957.0,32202.0,94759.0,498.75,8.0,22.0,0.36,32.0,93.22
Palacio,Entire home/apt,Entire rental unit,7+ guests,63,59,43516.0,8825.5,82592.5,329.5,8.0,17.0,0.21,40.0,93.65
Justicia,Entire home/apt,Entire rental unit,7+ guests,48,41,43500.0,14220.0,82485.0,496.8,8.0,7.5,0.40,28.0,85.42
Universidad,Entire home/apt,Entire rental unit,7+ guests,84,81,37260.0,9720.0,79292.0,426.0,8.0,15.0,0.26,54.0,96.43
Palacio,Entire home/apt,Entire rental unit,5-6 guests,211,189,34848.0,11394.0,59220.0,254.44,6.0,18.0,0.34,132.0,89.57
Sol,Entire home/apt,Entire rental unit,5-6 guests,170,151,31044.0,9960.0,55315.5,225.9,6.0,16.5,0.28,106.0,88.82
Guindalera,Entire home/apt,Entire rental unit,5-6 guests,41,38,30445.5,14919.0,45773.5,229.67,6.0,19.0,0.24,20.0,92.68
Embajadores,Entire home/apt,Entire rental unit,7+ guests,125,116,30435.5,7689.0,60796.75,368.34,8.0,9.0,0.27,60.0,92.8


listings  ...  revenue_coverage_percent
neighbourhood_cleansed room_type       property_type      capacity_band            ...                          
Sol                    Entire home/apt Entire rental unit 7+ guests            53  ...                     88.68
Cortes                 Entire home/apt Entire rental unit 7+ guests            38  ...                     92.11
Palacio                Entire home/apt Entire rental unit 7+ guests            50  ...                      94.0
Sol                    Entire home/apt Entire rental unit 5-6 guests          122  ...                     85.25
Universidad            Entire home/apt Entire rental unit 5-6 guests          146  ...                     85.62
                                                          7+ guests            62  ...                     95.16
Palacio                Entire home/apt Entire rental unit 5-6 guests          139  ...                     86.33
Embajadores            Entire home/apt Entire rental unit 7+ guests            91  ...                     93.41
Palos de Moguer        Entire home/apt Entire rental unit 1-2 guests           51  ...                     88.24
Cortes                 Entire home/apt Entire rental unit 5-6 guests          102  ...                     84.31
Sol                    Entire home/apt Entire rental unit 3-4 guests          257  ...                     92.61
Recoletos              Entire home/apt Entire rental unit 5-6 guests           47  ...                     95.74
Chopera                Entire home/apt Entire rental unit 3-4 guests           39  ...                     79.49
Argüelles              Entire home/apt Entire rental unit 3-4 guests           51  ...                     78.43
Cortes                 Entire home/apt Entire rental unit 3-4 guests          236  ...                     84.75

[15 rows x 11 columns]

### Summary calculated from the current CSV

The following output updates when the notebook is rerun. Revenue amounts are in the file's recorded local currency units. All reported associations are unadjusted unless explicitly described as a within-group comparison.

In [48]:
print(f'Revenue available for {len(revenue_observed):,} of {len(revenue_data):,} listings.')
print(f'Median estimated annual revenue: {revenue_observed[TARGET].median():,.1f}.')
print('Spearman associations with estimated revenue:')
for feature in ['estimated_occupancy_l365d', 'number_of_reviews_ltm', 'number_of_reviews', 'price']:
    row = formula_associations.loc[feature]
    print(f"  {feature}: {row['spearman']:.3f} (n={int(row['n']):,})")
print('Superhost group medians and observed-revenue counts:')
display(revenue_group_summaries['superhost_status'][['known_revenue', 'median_revenue']].round(2))
print('Categorical group separation: eta-squared on raw and log revenue')
display(category_associations[['eta_squared_raw_revenue', 'eta_squared_log_revenue']].round(4))
print('A high association with the constructed estimate does not establish a way to increase actual income.')

Revenue available for 19,172 of 22,708 listings.
Median estimated annual revenue: 8,401.5.
Spearman associations with estimated revenue:
  estimated_occupancy_l365d: 0.909 (n=19,172)
  number_of_reviews_ltm: 0.892 (n=19,172)
  number_of_reviews: 0.771 (n=19,172)
  price: 0.355 (n=19,172)
Superhost group medians and observed-revenue counts:
Categorical group separation: eta-squared on raw and log revenue
A high association with the constructed estimate does not establish a way to increase actual income.


,known_revenue,median_revenue
superhost_status,,
Superhost,5151,18916.0
Not Superhost,13889,5334.0
Unknown,132,1085.5


,eta_squared_raw_revenue,eta_squared_log_revenue
category,,
neighbourhood_cleansed,0.0484,0.0416
room_type,0.0580,0.0602
property_type,0.0605,0.0688
host_is_superhost,0.0469,0.0877
